# 灵活的Multi-Agent Propaganda检测系统（统一版）

## 核心特性

### ✨ 特性1: 自动发现语言和技术
- 添加新语言只需创建文件夹；添加新技术只需添加文件；零代码修改

### ✨ 特性2: 自动添加输出指令
- 系统自动添加输出格式要求（根据语言）；支持英语、俄语、波兰语

### ✨ 特性3: 语言感知的段落分割（EN/PO/RU 统一）
- 英语文章按**单换行** `\n` 分割（每行一个段落，符合 SemEval EN 格式）
- 俄语/波兰语按**双换行** `\n\n` 分割（空行区隔段落）
- 检测器从 `agent_system.language` 自动获取语言，无需手动指定

---

**运行顺序**: 步骤1–4（类定义）→ 步骤5（API配置）→ 步骤6（加载Prompt）→ 步骤8（初始化Agent）→ 步骤9（检测器）→ 步骤10（投票检测器）→ **加载数据** → **稳定版分批处理**（配置+运行）

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory of this repository
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 Task 3 development data
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirectories

# CheckThat! 2024 Task 3 data
CHECKTHAT_DIR = "your/path/here"

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## 步骤1: 导入必要的库

In [ ]:
import os
import json
import glob
from typing import List, Dict, Optional, Set, Tuple
from dataclasses import dataclass
from pathlib import Path
import re
from collections import Counter

from ollama import Client

from typing import Dict, List, Union
import time
from tqdm import tqdm
# AutoGen imports
from autogen import ConversableAgent, GroupChat, GroupChatManager

print("✓ All libraries imported successfully")

## 步骤2: 定义数据结构

In [ ]:
@dataclass
class TechniquePrompt:
    """说服技巧提示词数据结构"""
    name: str                    # 技术名称
    language: str                # 语言代码
    prompt_content: str          # 提示词内容
    file_path: str              # 文件路径
    category: Optional[str] = None  # 可选的分类

print("✓ Data structures defined")

## 步骤3: 创建灵活的提示词加载器

这个加载器会自动发现所有语言和技术

In [ ]:
class FlexiblePromptLoader:
    """灵活的提示词加载器 - 自动发现语言和技术"""
    
    def __init__(
        self,
        base_dir: str,
        languages: Optional[List[str]] = None,
        techniques: Optional[List[str]] = None,
        exclude_languages: Optional[List[str]] = None,
        exclude_techniques: Optional[List[str]] = None,
        config_file: Optional[str] = None,
        prompt_file_suffix: str = "_prompt.md",
        verbose: bool = True
    ):
        """
        初始化提示词加载器
        
        Args:
            base_dir: 提示词文件的基础目录
            languages: 指定要加载的语言列表（None表示加载所有）
            techniques: 指定要加载的技术列表（None表示加载所有）
            exclude_languages: 要排除的语言列表
            exclude_techniques: 要排除的技术列表
            config_file: 配置文件路径
            prompt_file_suffix: 提示词文件的后缀
            verbose: 是否显示详细信息
        """
        self.base_dir = base_dir
        self.prompt_file_suffix = prompt_file_suffix
        self.verbose = verbose
        
        # 如果提供了配置文件，从配置文件加载
        if config_file and os.path.exists(config_file):
            self._load_from_config(config_file)
        else:
            self.specified_languages = languages
            self.specified_techniques = techniques
            self.exclude_languages = exclude_languages or []
            self.exclude_techniques = exclude_techniques or []
        
        if not os.path.exists(base_dir):
            raise FileNotFoundError(f"提示词目录不存在: {base_dir}")
        
        # 存储加载的提示词
        self.prompts = {}  # {(technique, language): TechniquePrompt}
        self.discovered_languages = set()
        self.discovered_techniques = set()
        
        # 自动发现并加载
        self._discover_and_load()
    
    def _load_from_config(self, config_file: str):
        """从配置文件加载设置"""
        with open(config_file, 'r', encoding='utf-8') as f:
            config = json.load(f)
        
        self.specified_languages = config.get('languages')
        self.specified_techniques = config.get('techniques')
        self.exclude_languages = config.get('exclude_languages', [])
        self.exclude_techniques = config.get('exclude_techniques', [])
        
        if self.verbose:
            print(f"✓ 从配置文件加载: {config_file}")
    
    def _discover_languages(self) -> List[str]:
        """自动发现所有可用的语言"""
        languages = []
        for item in os.listdir(self.base_dir):
            item_path = os.path.join(self.base_dir, item)
            if os.path.isdir(item_path):
                # 检查目录中是否有提示词文件
                prompt_files = glob.glob(
                    os.path.join(item_path, f"*{self.prompt_file_suffix}")
                )
                if prompt_files:
                    languages.append(item.lower())
        return sorted(languages)
    
    def _discover_techniques_for_language(self, language: str) -> List[str]:
        """发现特定语言的所有技术"""
        techniques = []
        lang_dir = os.path.join(self.base_dir, language)
        
        if not os.path.exists(lang_dir):
            return []
        
        pattern = os.path.join(lang_dir, f"*{self.prompt_file_suffix}")
        prompt_files = glob.glob(pattern)
        
        for filepath in prompt_files:
            filename = os.path.basename(filepath)
            technique_name = filename[:-len(self.prompt_file_suffix)]
            techniques.append(technique_name)
        
        return sorted(techniques)
    
    def _should_include_language(self, language: str) -> bool:
        """判断是否应该加载某个语言"""
        if language in self.exclude_languages:
            return False
        if self.specified_languages is not None:
            return language in self.specified_languages
        return True
    
    def _should_include_technique(self, technique: str) -> bool:
        """判断是否应该加载某个技术"""
        if technique in self.exclude_techniques:
            return False
        if self.specified_techniques is not None:
            return technique in self.specified_techniques
        return True
    
    def _discover_and_load(self):
        """自动发现并加载所有提示词"""
        if self.verbose:
            print("\n" + "="*70)
            print("🔍 自动发现并加载提示词")
            print("="*70)
            print(f"基础目录: {self.base_dir}")
            print(f"提示词后缀: {self.prompt_file_suffix}")
        
        # 发现所有语言
        all_languages = self._discover_languages()
        
        if self.verbose:
            print(f"\n📂 发现的语言目录: {', '.join(all_languages) if all_languages else '无'}")
        
        if not all_languages:
            print("⚠️ 警告: 没有发现任何包含提示词文件的语言目录！")
            return
        
        # 过滤要加载的语言
        languages_to_load = [
            lang for lang in all_languages 
            if self._should_include_language(lang)
        ]
        
        if self.verbose and languages_to_load:
            print(f"✓ 将加载的语言: {', '.join(languages_to_load)}")
            if set(all_languages) - set(languages_to_load):
                excluded = set(all_languages) - set(languages_to_load)
                print(f"⊘ 排除的语言: {', '.join(excluded)}")
        
        loaded_count = 0
        total_techniques_set = set()
        
        # 对每个语言加载技术
        for language in languages_to_load:
            if self.verbose:
                print(f"\n📁 处理语言: {language.upper()}")
            
            all_techniques = self._discover_techniques_for_language(language)
            
            if self.verbose:
                print(f"  发现 {len(all_techniques)} 个技术文件")
            
            techniques_to_load = [
                tech for tech in all_techniques
                if self._should_include_technique(tech)
            ]
            
            if self.verbose and techniques_to_load:
                print(f"  将加载 {len(techniques_to_load)} 个技术")
                if set(all_techniques) - set(techniques_to_load):
                    excluded_count = len(set(all_techniques) - set(techniques_to_load))
                    print(f"  排除 {excluded_count} 个技术")
            
            # 加载每个技术的提示词
            for technique in techniques_to_load:
                filename = f"{technique}{self.prompt_file_suffix}"
                filepath = os.path.join(self.base_dir, language, filename)
                
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        content = f.read().strip()
                    
                    if content:
                        self.prompts[(technique, language)] = TechniquePrompt(
                            name=technique,
                            language=language,
                            prompt_content=content,
                            file_path=filepath
                        )
                        loaded_count += 1
                        total_techniques_set.add(technique)
                        self.discovered_languages.add(language)
                        self.discovered_techniques.add(technique)
                    else:
                        if self.verbose:
                            print(f"  ⚠️ 跳过空文件: {filename}")
                
                except Exception as e:
                    if self.verbose:
                        print(f"  ✗ 读取失败 {filename}: {e}")
        
        if self.verbose:
            print("\n" + "="*70)
            print("📊 加载总结")
            print("="*70)
            print(f"✓ 语言数量: {len(self.discovered_languages)}")
            print(f"  {', '.join(sorted(self.discovered_languages))}")
            print(f"\n✓ 技术数量: {len(total_techniques_set)}")
            techniques_list = sorted(total_techniques_set)
            for i in range(0, len(techniques_list), 3):
                batch = techniques_list[i:i+3]
                print(f"  {', '.join(batch)}")
            print(f"\n✓ 总计加载: {loaded_count} 个提示词文件")
            print("="*70 + "\n")
    
    def get_prompt(self, technique: str, language: str) -> Optional[TechniquePrompt]:
        """获取特定技术和语言的提示词"""
        return self.prompts.get((technique, language))
    
    def get_available_techniques(self, language: str) -> List[str]:
        """获取某个语言下所有可用的技术"""
        return sorted([
            tech for (tech, lang) in self.prompts 
            if lang == language
        ])
    
    def get_all_techniques(self) -> List[str]:
        """获取所有技术"""
        return sorted(list(self.discovered_techniques))
    
    def get_languages(self) -> List[str]:
        """获取所有语言"""
        return sorted(list(self.discovered_languages))
    
    def get_stats(self) -> Dict:
        """获取统计信息"""
        stats = {
            'total_prompts': len(self.prompts),
            'total_languages': len(self.discovered_languages),
            'total_techniques': len(self.discovered_techniques),
            'languages': sorted(list(self.discovered_languages)),
            'techniques': sorted(list(self.discovered_techniques)),
            'coverage': {}
        }
        
        for lang in self.discovered_languages:
            stats['coverage'][lang] = len(self.get_available_techniques(lang))
        
        return stats
    
    def save_config(self, output_path: str):
        """保存当前配置到文件"""
        config = {
            'base_dir': self.base_dir,
            'languages': sorted(list(self.discovered_languages)),
            'techniques': sorted(list(self.discovered_techniques)),
            'prompt_file_suffix': self.prompt_file_suffix,
            'stats': self.get_stats()
        }
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(config, f, ensure_ascii=False, indent=2)
        
        print(f"✓ 配置已保存到: {output_path}")

print("✓ FlexiblePromptLoader class defined")

## 步骤4: 创建灵活的Multi-Agent系统

**重点**: 这个类会自动在提示词后添加输出格式指令！

In [ ]:
class FlexibleTechniqueAgentSystem:
    """
    灵活的 Multi-Agent 说服技巧分类系统
    
    核心特性:
    1. 自动适应加载器中的语言和技术
    2. 自动在提示词后添加输出格式指令
    3. 根据语言选择合适的输出指令（俄语/波兰语/英语）
    """
    
    def __init__(
        self,
        prompt_loader,
        language: str,
        api_key: str,
        model: str = "gpt-4o-mini",
        temperature: float = 0,
        use_ollama: bool = False,
        ollama_host: str = "https://your-ollama-host/ollama"  # Set to your Ollama server URL
    ):
        """
        初始化Multi-Agent系统
        
        Args:
            prompt_loader: 提示词加载器实例
            language: 目标语言
            api_key: OpenAI API密钥
            model: 使用的模型
            temperature: 温度参数
            use_ollama: 是否使用Ollama
            ollama_host: Ollama服务器地址
        """
        self.prompt_loader = prompt_loader
        self.language = language.lower()
        self.model = model
        self.temperature = temperature
        self.use_ollama = use_ollama
        self.ollama_host = ollama_host
        
        # ✅ 这里需要4个空格的缩进（在 __init__ 方法内部）
        # 根据模型类型配置LLM
        if use_ollama:
            # Ollama配置
            self.ollama_client = Client(
                host=ollama_host,
                headers={'Authorization': f'Bearer {api_key}'}
            )
            self.llm_config = None
            self.api_key = api_key
            print(f"🔌 使用Ollama模型: {model}")
            print(f"🌐 服务器: {ollama_host}")
        else:
            # OpenAI配置
            self.llm_config = {
                "model": model,
                "api_key": api_key,
                "temperature": temperature
            }
            self.ollama_client = None
            print(f"🔌 使用OpenAI模型: {model}")
        
        # 获取该语言下可用的技术
        available_techniques = self.prompt_loader.get_available_techniques(self.language)
        
        if not available_techniques:
            raise ValueError(
                f"语言 '{self.language}' 没有可用的技术！\n"
                f"可用语言: {', '.join(self.prompt_loader.get_languages())}"
            )
        
        self.techniques = available_techniques
        
        print(f"\n🚀 初始化 {len(self.techniques)} 个技术的Agent系统")
        print(f"语言: {self.language.upper()}")
        print(f"模型: {model}")
        
        # 初始化agents
        self.agents = {}
        self._initialize_agents()
        
        print(f"✓ 所有agents初始化完成！\n")

    # ====================
    # 普通检测，无置信度，容易存在过度检测
    # ====================
    # def _get_output_instruction(self, technique_name: str) -> str:
    #     """
    #     获取输出格式指令（英语版本）
        
    #     Args:
    #         technique_name: 技术名称
        
    #     Returns:
    #         输出格式指令字符串
    #     """
    #     return f"""
    # {'='*70}
    # CRITICAL OUTPUT INSTRUCTIONS:
    # {'='*70}

    # Your ONLY task is to determine if the technique "{technique_name}" is present in the given text.

    # RESPONSE FORMAT (STRICT):
    # 1. You MUST respond with ONLY '1' or '0'
    # 2. '1' = technique "{technique_name}" IS present in the text
    # 3. '0' = technique "{technique_name}" IS NOT present in the text
    # 4. NO explanations, NO justifications, NO additional text whatsoever
    # 5. If uncertain, respond with '0'

    # Your response must be exactly one character: either '1' or '0'.

    # EXAMPLES OF CORRECT RESPONSES:
    # - 1
    # - 0

    # EXAMPLES OF INCORRECT RESPONSES:
    # - 1 (technique is present)  ← DO NOT add explanations
    # - Yes, it's present  ← ONLY '1' or '0'
    # - 0, because...  ← DO NOT add justification

    # RESPOND WITH ONLY '1' OR '0':
    # """

    def _get_output_instruction(self, technique_name: str) -> str:
        """获取更严格的输出格式指令"""
        return f"""
    {'='*70}
    CRITICAL OUTPUT INSTRUCTIONS:
    {'='*70}

    Your task is to determine if the technique "{technique_name}" is CLEARLY and SIGNIFICANTLY present in the TARGET PARAGRAPH.

    STRICT EVALUATION CRITERIA:
    1. The technique must be EXPLICIT and OBVIOUS in the target paragraph
    2. There must be CLEAR EVIDENCE, not just vague similarities
    3. The technique must be CENTRAL to the paragraph's message, not just tangentially related
    4. When in doubt or if the evidence is weak, respond with '0'
    5. Only respond '1' if you are HIGHLY CONFIDENT the technique is present

    RESPONSE FORMAT:
    - '1' = technique "{technique_name}" is CLEARLY and SIGNIFICANTLY present
    - '0' = technique is absent OR evidence is weak/uncertain

    Be CONSERVATIVE in your judgment. It's better to miss a technique than to falsely detect one.

    Your response must be exactly one character: either '1' or '0'.

    RESPOND WITH ONLY '1' OR '0':
    """
    
    def _create_agent_system_prompt(self, technique_name: str) -> str:
        """
        为单个技巧创建agent的系统提示词
        
        工作流程:
        1. 读取用户编写的技术描述提示词
        2. 根据语言获取合适的输出格式指令
        3. 将两者组合成完整的系统提示词
        
        这样用户的提示词文件只需要包含技术描述，不需要写输出格式！
        
        Args:
            technique_name: 技术名称
        
        Returns:
            完整的系统提示词（技术描述 + 输出指令）
        """
        # 1. 获取技术提示词对象
        technique_prompt = self.prompt_loader.get_prompt(technique_name, self.language)
        
        if not technique_prompt:
            raise ValueError(
                f"找不到技巧 {technique_name} 在语言 {self.language} 下的提示词"
            )
        
        # 2. 获取原始提示词内容（用户编写的技术描述）
        original_content = technique_prompt.prompt_content
        
        # 3. 获取适合该语言的输出格式指令
        output_instruction = self._get_output_instruction(technique_name)
        
        # 4. 组合完整的系统提示词
        # 原始技术描述 + 空行 + 输出格式指令
        full_prompt = f"""{original_content}

{output_instruction}"""
        
        return full_prompt
    
    def _initialize_agents(self):
        """初始化所有技术的agents"""
        for technique in self.techniques:
            # 创建完整的系统提示词（自动添加输出指令）
            system_message = self._create_agent_system_prompt(technique)
            if self.use_ollama:
                # 对于Ollama，我们不创建AutoGen agent
                # 而是保存system message供后续使用
                self.agents[technique] = {
                    'system_message': system_message,
                    'type': 'ollama'
                }
            else:
                # 创建标准的AutoGen agent
                agent = ConversableAgent(
                    name=f"{technique}_agent",
                    system_message=system_message,
                    llm_config=self.llm_config,
                    human_input_mode="NEVER",
                    max_consecutive_auto_reply=1
                )
                self.agents[technique] = agent
    
    def classify_text(self, text: str, verbose: bool = False) -> Tuple[List[str], Dict]:
        """
        对文本进行分类
        
        Args:
            text: 要分类的文本
            verbose: 是否显示详细信息
        
        Returns:
            (detected_techniques, chat_history)
            - detected_techniques: 检测到的技术列表
            - chat_history: 完整的对话历史
        """
        detected_techniques = []
        chat_history = {}
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"开始分类 ({len(self.techniques)} 个技术)")
            print(f"{'='*70}\n")
        
        for i, technique in enumerate(self.techniques, 1):
            if verbose:
                print(f"[{i}/{len(self.techniques)}] 检查: {technique}...", end=" ")
            
            try:
                agent = self.agents[technique]

                if self.use_ollama:
                    system_message = agent['system_message']
            
                    response = self.ollama_client.chat(
                        model=self.model,
                        messages=[
                            {'role': 'system', 'content': system_message},
                            {'role': 'user', 'content': text}
                        ]
                    )
                    # 提取Ollama响应
                    result = response['message']['content'].strip()
                else:
                    # 使用AutoGen agent
                    response = agent.generate_reply(
                        messages=[{"role": "user", "content": text}]
                    )
            
                    # 🔧 修复：处理None响应
                    if response is None:
                        # generate_reply可能返回None，这时使用默认值
                        result = '0'
                        if verbose:
                            print("⚠️ (Agent返回None，默认为0) ", end="")
                    elif isinstance(response, str):
                        result = response.strip()
                    else:
                        result = str(response).strip()
                
                # 🔧 修复：更宽松的检测逻辑
                # 即使响应不是纯'0'/'1'，也尝试提取
                result_clean = result.strip()
                
                # 保存历史
                chat_history[technique] = {
                    'response': result_clean,
                    'detected': result_clean == '1'
                }

                # 检查是否检测到
                if result_clean == '1':
                    detected_techniques.append(technique)
                    if verbose:
                        print("✓ 检测到")
                elif result_clean == '0':
                    if verbose:
                        print("✗ 未检测到")
                else:
                    # 响应格式不对，尝试提取
                    if '1' in result_clean and '0' not in result_clean:
                        detected_techniques.append(technique)
                        if verbose:
                            print(f"⚠️ 检测到 (格式异常: '{result_clean[:30]}')")
                    else:
                        if verbose:
                            print(f"⚠️ 未检测到 (格式异常: '{result_clean[:30]}')")

            except Exception as e:
                if verbose:
                    print(f"❌ 错误: {e}")
                chat_history[technique] = {
                    'response': None,
                    'error': str(e),
                    'detected': False
                }
                
        if verbose:
            print(f"\n{'='*70}")
            print(f"分类完成！检测到 {len(detected_techniques)} 个技术")
            if detected_techniques:
                print(f"检测到的技术: {', '.join(detected_techniques)}")
            print(f"{'='*70}\n")
        
        return detected_techniques, chat_history
    def classify_batch(
        self,
        fragments: List[Dict],
        text_key: str = 'text',
        verbose: bool = True
    ) -> List[Dict]:
        """
        批量分类多个文本片段
        
        Args:
            fragments: 文本片段列表，每个是包含文本的字典
            text_key: 文本字段的键名
            verbose: 是否显示进度
        
        Returns:
            结果列表，每个包含原始数据和检测结果
        """
        results = []
        total = len(fragments)
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"开始批量分类: {total} 个文本片段")
            print(f"{'='*70}\n")
        
        for i, fragment in enumerate(fragments, 1):
            if verbose:
                print(f"\n处理片段 {i}/{total}...")
            
            text = fragment.get(text_key, '')
            
            if not text:
                if verbose:
                    print("  ⚠️ 跳过空文本")
                results.append({
                    **fragment,
                    'detected_techniques': [],
                    'chat_history': {},
                    'error': 'Empty text'
                })
                continue
            
            try:
                detected, history = self.classify_text(text, verbose=False)
                
                results.append({
                    **fragment,
                    'detected_techniques': detected,
                    'chat_history': history
                })
                
                if verbose:
                    print(f"  ✓ 检测到 {len(detected)} 个技术")
                    if detected:
                        print(f"    {', '.join(detected)}")
            
            except Exception as e:
                if verbose:
                    print(f"  ✗ 错误: {e}")
                results.append({
                    **fragment,
                    'detected_techniques': [],
                    'chat_history': {},
                    'error': str(e)
                })
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"批量分类完成！")
            print(f"{'='*70}\n")
        
        return results

print("✓ FlexibleTechniqueAgentSystem类定义完成！")

In [ ]:
# ============================================================
# 带上下文的段落检测器（EN/PO/RU 统一，语言感知分割）
# ============================================================

from typing import List, Dict, Tuple
import json as _json

class ContextAwareParagraphDetector:
    """
    带上下文的段落级 propaganda 检测器（EN/PO/RU 统一版）

    段落分割策略（由 self.language 自动选择）:
      - 'en': text.split('\\n')            每行一段（SemEval EN 格式）
      - 'po'/'ru': re.split(r'\\n\\s*\\n') 空行区隔段落

    语言从 agent_system.language 自动获取。
    """

    def __init__(self, agent_system, language: str = None):
        self.agent_system = agent_system

        # 自动获取语言（优先从 agent_system 读取）
        if language is None and hasattr(agent_system, 'language'):
            self.language = agent_system.language.lower()
        else:
            self.language = language.lower() if language else 'en'

        print(f"✓ 检测器初始化完成 (语言: {self.language.upper()})")

    def split_into_paragraphs(self, text: str, min_length: int = None) -> List[Dict]:
        """
        将文本分割成段落（语言感知版）

        Args:
            text:       原始文本
            min_length: 最小段落长度（字符数）。
                        None = 自动：EN 为 1（保留短行），PO/RU 为 50。
        """
        # 自动 min_length
        if min_length is None:
            min_length = 1 if self.language == 'en' else 50

        paragraphs = []

        # ── 语言感知分割 ──────────────────────────────────────────────────
        if self.language == 'en':
            # 英语：SemEval 格式每行即一个段落
            raw_paragraphs = text.split('\n')
        else:
            # 波兰语 / 俄语：段落间以空行区隔
            raw_paragraphs = re.split(r'\n\s*\n', text)

        current_pos = 0

        for line_number, raw_para in enumerate(raw_paragraphs, start=1):
            raw_para = raw_para.strip()

            if len(raw_para) < min_length:
                pos = text.find(raw_para, current_pos)
                if pos != -1:
                    current_pos = pos + len(raw_para)
                continue

            start_pos = text.find(raw_para, current_pos)
            if start_pos == -1:
                continue

            end_pos = start_pos + len(raw_para) - 1

            paragraphs.append({
                'paragraph_id': line_number,
                'start_pos':    start_pos,
                'end_pos':      end_pos,
                'text':         raw_para,
                'char_count':   len(raw_para),
                'word_count':   len(raw_para.split())
            })

            current_pos = end_pos + 1

        return paragraphs

    def create_context_prompt(
        self,
        full_article: str,
        target_paragraph: str,
        paragraph_id: int,
        total_paragraphs: int
    ) -> str:
        """在完整文章中标记目标段落，构造 LLM 输入提示"""
        marked_article = full_article.replace(
            target_paragraph,
            f"\n{'='*70}\n>>> TARGET PARAGRAPH (段落 {paragraph_id}/{total_paragraphs}) <<<\n{'='*70}\n{target_paragraph}\n{'='*70}\n>>> END OF TARGET PARAGRAPH <<<\n{'='*70}\n",
            1
        )
        prompt = f"""CONTEXT: You are analyzing paragraph {paragraph_id} out of {total_paragraphs} paragraphs in the following article.

FULL ARTICLE (for context):
{'─'*70}
{marked_article}
{'─'*70}

IMPORTANT INSTRUCTIONS:
1. The article above is provided as CONTEXT to help you understand the overall narrative, tone, and argumentation strategy.
2. Your task is to determine if the propaganda technique is present in the marked TARGET PARAGRAPH ONLY (between the === markers).
3. You may use the full article context to better understand the TARGET PARAGRAPH, but you should ONLY evaluate whether the technique appears in the TARGET PARAGRAPH itself.
4. Consider the context when making your judgment, but base your decision on the TARGET PARAGRAPH content.

TARGET PARAGRAPH TO ANALYZE:
{target_paragraph}
"""
        return prompt

    def detect_paragraph_with_context(
        self,
        full_article: str,
        target_paragraph: str,
        paragraph_id: int,
        total_paragraphs: int,
        verbose: bool = False
    ) -> Tuple[List[str], Dict]:
        """在完整文章上下文中检测单个段落"""
        context_prompt = self.create_context_prompt(
            full_article, target_paragraph, paragraph_id, total_paragraphs
        )
        return self.agent_system.classify_text(context_prompt, verbose=verbose)

    def detect_article_by_paragraphs(
        self,
        article_id: str,
        full_text: str,
        min_paragraph_length: int = None,
        verbose: bool = True
    ) -> Dict:
        """按段落检测整篇文章（带上下文）"""
        if verbose:
            print(f"\n{'='*70}")
            print(f"📄 分析文章: {article_id}  [语言: {self.language.upper()}]")
            print(f"{'='*70}")

        paragraphs      = self.split_into_paragraphs(full_text, min_paragraph_length)
        total_paragraphs = len(paragraphs)

        if verbose:
            print(f"📊 文章信息:")
            print(f"  - 总字符数: {len(full_text):,}")
            print(f"  - 段落数:   {total_paragraphs}")
            if paragraphs:
                avg = sum(p['char_count'] for p in paragraphs) / total_paragraphs
                print(f"  - 平均段落长度: {avg:.0f} 字符")
            print(f"\n🔍 开始段落级检测（每个段落都会看到完整文章上下文）...")

        results = []
        for i, para in enumerate(paragraphs, 1):
            if verbose:
                print(f"\n{'─'*70}")
                print(f"[段落 {i}/{total_paragraphs}]  位置: {para['start_pos']}-{para['end_pos']}  长度: {para['char_count']} 字符")
                print(f"  预览: {para['text'][:100]}...")

            detected, responses = self.detect_paragraph_with_context(
                full_article=full_text,
                target_paragraph=para['text'],
                paragraph_id=para['paragraph_id'],
                total_paragraphs=total_paragraphs,
                verbose=False
            )

            results.append({
                'article_id':          article_id,
                'paragraph_id':        para['paragraph_id'],
                'start_pos':           para['start_pos'],
                'end_pos':             para['end_pos'],
                'char_count':          para['char_count'],
                'detected_techniques': detected,
                'num_techniques':      len(detected),
                'text':                para['text'],
                'text_preview':        para['text'][:150]
            })

            if verbose:
                if detected:
                    print(f"  ✓ 检测到 {len(detected)} 个技术: {', '.join(detected)}")
                else:
                    print(f"  ○ 未检测到 propaganda 技术")

        if verbose:
            total_tech   = sum(r['num_techniques'] for r in results)
            paras_w_tech = sum(1 for r in results if r['num_techniques'] > 0)
            print(f"\n{'='*70}")
            print(f"✅ 文章分析完成  段落: {len(results)}  有技术段落: {paras_w_tech}  技术实例: {total_tech}")
            print(f"{'='*70}\n")

        return {
            'article_id':        article_id,
            'total_paragraphs':  total_paragraphs,
            'full_text_length':  len(full_text),
            'full_text':         full_text,
            'paragraph_results': results
        }

    def format_output_tsv(self, detection_result: Dict) -> List[str]:
        """格式化为 TSV 行: article_id \t start_pos \t end_pos \t technique..."""
        lines = []
        aid = detection_result['article_id']
        for pr in detection_result['paragraph_results']:
            if pr['detected_techniques']:
                lines.append(f"{aid}\t{pr['start_pos']}\t{pr['end_pos']}\t{'\t'.join(pr['detected_techniques'])}")
        return lines

    def save_results_tsv(self, detection_results: List[Dict], output_file: str):
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("article_id\tstart_pos\tend_pos\ttechniques\n")
            for res in detection_results:
                for line in self.format_output_tsv(res):
                    f.write(line + '\n')
        print(f"✓ TSV 结果已保存: {output_file}")

    def save_results_detailed(self, detection_results: List[Dict], output_file: str):
        total_p   = sum(r['total_paragraphs'] for r in detection_results)
        tech_inst = sum(pr['num_techniques'] for r in detection_results for pr in r['paragraph_results'])
        tech_counter = {}
        for r in detection_results:
            for pr in r['paragraph_results']:
                for t in pr['detected_techniques']:
                    tech_counter[t] = tech_counter.get(t, 0) + 1
        output_data = {
            'summary': {
                'total_articles': len(detection_results),
                'total_paragraphs': total_p,
                'total_technique_instances': tech_inst,
                'technique_frequency': dict(sorted(tech_counter.items(), key=lambda x: x[1], reverse=True))
            },
            'detection_method': 'context-aware paragraph detection (unified EN/PO/RU)',
            'articles': detection_results
        }
        with open(output_file, 'w', encoding='utf-8') as f:
            _json.dump(output_data, f, ensure_ascii=False, indent=2)
        print(f"✓ 详细 JSON 结果已保存: {output_file}")


print("✓ ContextAwareParagraphDetector 类定义完成（EN/PO/RU 统一版）")


In [ ]:
# ============================================================
# 基于ContextAwareParagraphDetector的投票检测器
# ============================================================


class VotingContextAwareParagraphDetector(ContextAwareParagraphDetector):
    """
    带投票机制的段落检测器
    """
    
    def __init__(
        self, 
        agent_system: FlexibleTechniqueAgentSystem,
        voting_rounds: int = 3,
        voting_threshold: float = 0.6,
        use_temperature: bool = True,
        temperature: float = 0.3
    ):
        super().__init__(agent_system)
        self.voting_rounds = voting_rounds
        self.voting_threshold = voting_threshold
        self.use_temperature = use_temperature
        self.temperature = temperature
        
        if use_temperature:
            self.original_temperature = agent_system.temperature
            agent_system.temperature = temperature
    
    def detect_paragraph_with_voting(
        self,
        full_article: str,
        target_paragraph: str,
        paragraph_id: int,
        total_paragraphs: int,
        verbose: bool = False
    ) -> Dict:
        """使用投票机制检测单个段落"""
        if verbose:
            print(f"\n🗳️  开始投票检测 ({self.voting_rounds}轮)...")
        
        all_round_results = []
        technique_vote_counts = Counter()
        
        # 多轮检测
        for round_num in range(self.voting_rounds):
            if verbose:
                print(f"  轮次 {round_num + 1}/{self.voting_rounds}...", end=" ")
            
            # 调用父类方法
            detected_techniques, _ = self.detect_paragraph_with_context(
                full_article=full_article,
                target_paragraph=target_paragraph,
                paragraph_id=paragraph_id,
                total_paragraphs=total_paragraphs,
                verbose=False
            )
            
            all_round_results.append(detected_techniques)
            
            for tech in detected_techniques:
                technique_vote_counts[tech] += 1
            
            if verbose:
                print(f"检测到 {len(detected_techniques)} 个技术")
            
            if round_num < self.voting_rounds - 1:
                time.sleep(0.5)
        
        # 投票结果
        min_votes = int(self.voting_rounds * self.voting_threshold)
        final_techniques = [
            tech for tech, count in technique_vote_counts.items()
            if count >= min_votes
        ]
        
        final_techniques.sort(
            key=lambda t: technique_vote_counts[t],
            reverse=True
        )
        
        if verbose and final_techniques:
            print(f"\n  📊 投票统计:")
            print(f"     阈值: {min_votes}/{self.voting_rounds} 轮")
            print(f"     通过技术数: {len(final_techniques)}")
            for tech in final_techniques:
                votes = technique_vote_counts[tech]
                print(f"       • {tech}: {votes}/{self.voting_rounds} 轮 ({votes/self.voting_rounds:.0%})")
        
        return {
            'detected_techniques': final_techniques,
            'num_techniques': len(final_techniques),
            'voting_details': {
                'rounds': self.voting_rounds,
                'threshold': self.voting_threshold,
                'all_round_results': all_round_results,
                'vote_counts': dict(technique_vote_counts),
                'passed_techniques': final_techniques
            }
        }
    
    def detect_article_by_paragraphs(
        self,
        article_id: str,
        full_text: str,
        min_paragraph_length: int = 50,
        verbose: bool = False
    ) -> Dict:
        """使用投票机制检测整篇文章"""
        if verbose:
            print(f"\n{'='*70}")
            print(f"📄 分析文章（投票模式）: {article_id}")
            print(f"{'='*70}")
            print(f"⚙️  投票设置:")
            print(f"   - 轮数: {self.voting_rounds}")
            print(f"   - 阈值: {self.voting_threshold} ({int(self.voting_rounds * self.voting_threshold)}/{self.voting_rounds}轮)")
            if self.use_temperature:
                print(f"   - 温度: {self.temperature}")
        
        # 分割段落
        paragraphs = self.split_into_paragraphs(full_text, min_paragraph_length)
        total_paragraphs = len(paragraphs)
        
        if verbose:
            print(f"📊 文章信息:")
            print(f"  - 总字符数: {len(full_text):,}")
            print(f"  - 段落数: {total_paragraphs}")
            if paragraphs:
                avg_len = sum(p['char_count'] for p in paragraphs) // total_paragraphs
                print(f"  - 平均段落长度: {avg_len} 字符")
            print(f"\n🔍 开始段落级投票检测...")
        
        paragraph_results = []
        
        for i, para_info in enumerate(paragraphs, 1):
            if verbose:
                print(f"\n{'─'*70}")
                print(f"[段落 {i}/{total_paragraphs}]")
                print(f"  位置: {para_info['start_pos']}-{para_info['end_pos']}")
                print(f"  长度: {para_info['char_count']} 字符")
                print(f"  预览: {para_info['text'][:70]}...")
            
            # 使用投票检测
            result = self.detect_paragraph_with_voting(
                full_article=full_text,
                target_paragraph=para_info['text'],
                paragraph_id=para_info['paragraph_id'],
                total_paragraphs=total_paragraphs,
                verbose=verbose
            )
            
            # 添加位置信息
            result['article_id'] = article_id
            result['start_pos'] = para_info['start_pos']
            result['end_pos'] = para_info['end_pos']
            result['paragraph_text'] = para_info['text']
            
            paragraph_results.append(result)
            
            if result['num_techniques'] > 0:
                if verbose:
                    print(f"  ✓ 最终确认 {result['num_techniques']} 个技术:")
                    for tech in result['detected_techniques']:
                        votes = result['voting_details']['vote_counts'][tech]
                        print(f"    • {tech} ({votes}/{self.voting_rounds})")
            else:
                if verbose:
                    print(f"  ○ 未检测到propaganda技术")
        
        total_techniques = sum(r['num_techniques'] for r in paragraph_results)
        paragraphs_with_techniques = sum(1 for r in paragraph_results if r['num_techniques'] > 0)
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✅ 文章分析完成")
            print(f"  总段落数: {len(paragraph_results)}")
            print(f"  有技术的段落: {paragraphs_with_techniques} ({paragraphs_with_techniques/len(paragraph_results):.1%})")
            print(f"  总技术实例数: {total_techniques}")
            print(f"{'='*70}")
        
        return {
            'article_id': article_id,
            'full_text': full_text,
            'num_paragraphs': len(paragraph_results),
            'paragraphs_with_techniques': paragraphs_with_techniques,
            'total_techniques': total_techniques,
            'paragraph_results': paragraph_results,
            'voting_config': {
                'rounds': self.voting_rounds,
                'threshold': self.voting_threshold,
                'temperature': self.temperature if self.use_temperature else 0.0
            }
        }
    
    def __del__(self):
        if hasattr(self, 'use_temperature') and self.use_temperature:
            if hasattr(self, 'original_temperature'):
                self.agent_system.temperature = self.original_temperature


print("✓ VotingContextAwareParagraphDetector 类已更新（使用正确的方法签名）")

## 步骤5: 配置API密钥和路径

**请修改下面的路径为你的实际路径！**

In [ ]:
# ========================================
# 配置部分
# ========================================

# 选择使用哪个系统
USE_OLLAMA = False  # True=Ollama, False=OpenAI

# 路径配置
API_CONFIG_PATH = "your/path/here"  # Update to your local path
PROMPTS_BASE_DIR = "your/path/here"  # Update to your local path
# DATA_BASE_DIR = "your/path/here"  # Update to your local path
DATA_CONFIGS = {
    'checkthat_base': 'your/path/here'  # CheckThat! 2024 data directory,
    'train_base': 'your/path/here'  # Update to your local path,
    'trial_base': 'your/path/here'  # Update to your local path  # 保留旧路径以防需要
}

# 加载API配置
print("加载API配置...")
try:
    with open(API_CONFIG_PATH, 'r') as f:
        api_config = json.load(f)
    
    # ✅ 从 api_keys 对象中获取密钥
    api_keys = api_config.get('api_keys', {})
    
    if USE_OLLAMA:
        # 使用Ollama
        api_key = api_keys.get('ollama_api_key')
        
        # 如果没有ollama_api_key，尝试使用openai_api_key
        if not api_key:
            api_key = api_keys.get('openai_api_key')
            print("⚠️ 未找到ollama_api_key，使用openai_api_key替代")
        
        model = 'llama3:70b'  # 或其他Ollama模型
        print("✓ 配置为使用Ollama模型")
        print(f"  模型: {model}")
    else:
        # 使用OpenAI
        api_key = api_keys.get('openai_api_key')
        model = 'gpt-4o-mini'
        print("✓ 配置为使用OpenAI模型")
        print(f"  模型: {model}")
    
    if not api_key:
        raise ValueError(f"未找到{'Ollama' if USE_OLLAMA else 'OpenAI'}的API密钥！")
    
    print("✓ API密钥加载成功！")
    
except Exception as e:
    print(f"✗ 加载API配置失败: {e}")
    print("请确保路径正确并且文件存在！")
    api_key = None
    model = None

## 步骤6: 初始化提示词加载器

这一步会自动发现所有语言和技术

In [ ]:
# 初始化加载器（自动发现所有内容）
loader = FlexiblePromptLoader(
    base_dir=PROMPTS_BASE_DIR,
    languages=['ru', 'po','en'],  # 只加载俄语和波兰语
    # exclude_techniques=['Slogans'],  # 可选：排除某些技术
    verbose=True
)

## 步骤7: 查看加载的统计信息

In [ ]:
# 获取统计信息
stats = loader.get_stats()

print("\n📊 统计信息:")
print(f"总提示词数: {stats['total_prompts']}")
print(f"语言数: {stats['total_languages']}")

print(f"技术数: {stats['total_techniques']}")

print("\n📂 每种语言的覆盖情况:")
for lang, count in stats['coverage'].items():
    print(f"  {lang.upper()}: {count} 个技术")

## 步骤8: 测试输出指令生成

**重要**: 这里可以看到系统如何自动添加输出指令！

In [ ]:
# 查看某个技术的原始提示词

import warnings
import logging

# 抑制AutoGen的API密钥格式警告
warnings.filterwarnings('ignore', message='The API key specified is not a valid OpenAI format')

# 或者设置autogen的logging级别
logging.getLogger('autogen.oai.client').setLevel(logging.ERROR)

prompt = loader.get_prompt('Straw_Man', 'po')
# print(prompt.prompt_content)  # ← 这是从.md文件读取的内容

# 查看完整的系统提示词(包含输出指令)
agent_system = FlexibleTechniqueAgentSystem(
    prompt_loader=loader,
    language='po',
    api_key=api_key,
    model=model
)
full_prompt = agent_system._create_agent_system_prompt('Straw_Man')
print(full_prompt)  # ← 文件内容 + 输出指令

## 步骤9: 初始化Agent系统

系统会自动为每个技术添加输出指令！

In [ ]:

import logging

# 设置autogen的日志级别为ERROR，忽略WARNING
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)


#初始化波兰语
# 检查API密钥是否可用
if api_key is None:
    print("⚠️ API密钥未加载，无法初始化系统！")
    print("请先运行步骤5并确保API配置正确。")
else:
    # 初始化波兰语系统
    system_po = FlexibleTechniqueAgentSystem(
        prompt_loader=loader,
        language='po',  # 波兰语
        api_key=api_key,
        model='gpt-4o-mini',
        temperature=0
    )

if api_key:
    # 初始化俄语系统
    system_ru = FlexibleTechniqueAgentSystem(
        prompt_loader=loader,
        language='ru',  # 俄语
        api_key=api_key,
        model='gpt-4o-mini',
        temperature=0
    )

    system_en = FlexibleTechniqueAgentSystem(
        prompt_loader=loader,
        language='en',  # 英语 (English)
        api_key=api_key,
        model='gpt-4o-mini',
        temperature=0
    )
    # 查看俄语的输出指令
    # ru_prompt = system_ru._create_agent_system_prompt("Straw_Man")
    
    # print("\n俄语系统的输出指令（后500字符）:")
    # print("="*70)
    # print(ru_prompt[-500:])
    # print("\n✓ 可以看到俄语指令与波兰语不同！")
else:
    print("⚠️ 请先加载API密钥（运行步骤5）")

In [ ]:
# ============================================================
# 为每种语言创建专门的检测器
# ============================================================

print("="*70)
print("🚀 初始化带上下文的段落检测器（按语言分离）")
print("="*70)

# 俄语检测器
context_detector_ru = ContextAwareParagraphDetector(system_ru)
print("✓ 俄语检测器创建完成")

# 波兰语检测器
context_detector_po = ContextAwareParagraphDetector(system_po)
print("✓ 波兰语检测器创建完成")

# 英语检测器
context_detector_en = ContextAwareParagraphDetector(system_en)
print("✓ 英语检测器创建完成")

# 保留旧的变量名（用于向后兼容）
context_detector = context_detector_po  # 默认使用波兰语

print("\n可用的检测器:")
print("  - context_detector_ru (俄语)")
print("  - context_detector_po (波兰语)")
print("  - context_detector_en (英语)")
print("="*70)

In [ ]:
import logging

# 设置autogen的日志级别为ERROR,忽略WARNING
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

# ============================================================
# 为每种语言创建投票检测器
# ============================================================

print("\n" + "="*70)
print("🗳️  初始化投票检测器(按语言分离)")
print("="*70)

# ============================================================
# 俄语投票检测器
# ============================================================

print("\n[1/3] 创建俄语投票检测器...")

# 平衡模式
voting_detector_ru_balanced = VotingContextAwareParagraphDetector(
    agent_system=system_ru,
    voting_rounds=3,
    voting_threshold=0.67,
    use_temperature=True,
    temperature=0.3
)

# 保守模式
voting_detector_ru_conservative = VotingContextAwareParagraphDetector(
    agent_system=system_ru,
    voting_rounds=5,
    voting_threshold=0.8,
    use_temperature=True,
    temperature=0.3
)

# 激进模式
voting_detector_ru_aggressive = VotingContextAwareParagraphDetector(
    agent_system=system_ru,
    voting_rounds=3,
    voting_threshold=0.34,
    use_temperature=True,
    temperature=0.4
)

print("✓ 俄语投票检测器创建完成(3种模式)")

# ============================================================
# 波兰语投票检测器
# ============================================================

print("\n[2/3] 创建波兰语投票检测器...")

# 平衡模式
voting_detector_po_balanced = VotingContextAwareParagraphDetector(
    agent_system=system_po,
    voting_rounds=3,
    voting_threshold=0.67,
    use_temperature=True,
    temperature=0.3
)

# 保守模式
voting_detector_po_conservative = VotingContextAwareParagraphDetector(
    agent_system=system_po,
    voting_rounds=5,
    voting_threshold=0.8,
    use_temperature=True,
    temperature=0.3
)

# 激进模式
voting_detector_po_aggressive = VotingContextAwareParagraphDetector(
    agent_system=system_po,
    voting_rounds=3,
    voting_threshold=0.34,
    use_temperature=True,
    temperature=0.4
)

print("✓ 波兰语投票检测器创建完成(3种模式)")

# ============================================================
# 英语投票检测器
# ============================================================

print("\n[3/3] 创建英语投票检测器...")

# 平衡模式
voting_detector_en_balanced = VotingContextAwareParagraphDetector(
    agent_system=system_en,
    voting_rounds=3,
    voting_threshold=0.67,
    use_temperature=True,
    temperature=0.3
)

# 保守模式
voting_detector_en_conservative = VotingContextAwareParagraphDetector(
    agent_system=system_en,
    voting_rounds=5,
    voting_threshold=0.8,
    use_temperature=True,
    temperature=0.3
)

# 激进模式
voting_detector_en_aggressive = VotingContextAwareParagraphDetector(
    agent_system=system_en,
    voting_rounds=3,
    voting_threshold=0.34,
    use_temperature=True,
    temperature=0.4
)

print("✓ 英语投票检测器创建完成(3种模式)")

# 保留旧的变量名(向后兼容)
voting_detector_balanced = voting_detector_po_balanced
voting_detector_conservative = voting_detector_po_conservative
voting_detector_aggressive = voting_detector_po_aggressive

print("\n✓ 所有投票检测器初始化完成!")
print("\n可用的检测器:")
print("  俄语: voting_detector_ru_{balanced/conservative/aggressive}")
print("  波兰语: voting_detector_po_{balanced/conservative/aggressive}")
print("  英语: voting_detector_en_{balanced/conservative/aggressive}")
print("="*70)

##  加载真实标注和评估器

In [ ]:
import os
from typing import List, Dict, Set, Tuple
from collections import defaultdict

# ============================================================
# 真实标注加载和评估系统
# ============================================================

class PropagandaEvaluator:
    """Propaganda检测评估器"""
    
    def __init__(self):
        # 技术名称标准化映射
        self.technique_normalization = {
            'Appeal_to_Fear-Prejudice': 'Appeal_to_Fear_Prejudice',
            'Repetitions': 'Repetition',
            'Name_Calling-Labeling': 'Name_Calling_Labeling',
            'Exaggeration-Minimisation': 'Exaggeration_Minimisation',
            # 添加其他可能的变体
        }
    
    def normalize_technique_name(self, technique: str) -> str:
        """
        标准化技术名称
        
        Args:
            technique: 原始技术名称
        
        Returns:
            标准化后的技术名称
        """
        # 使用映射表
        if technique in self.technique_normalization:
            return self.technique_normalization[technique]
        
        # 替换连字符为下划线
        technique = technique.replace('-', '_')
        
        return technique
    
    def load_gold_annotations(self, annotation_file: str) -> Dict[str, List[Dict]]:
        """
        加载真实标注文件
        
        Args:
            annotation_file: 标注文件路径
        
        Returns:
            字典：{article_id: [annotation1, annotation2, ...]}
        """
        annotations = defaultdict(list)
        
        if not os.path.exists(annotation_file):
            print(f"✗ 标注文件不存在: {annotation_file}")
            return {}
        
        with open(annotation_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split('\t')
                if len(parts) < 4:  # 至少需要：article_id, start, end, technique
                    continue
                
                article_id = parts[0]
                start_pos = int(parts[1])
                end_pos = int(parts[2])
                techniques = [self.normalize_technique_name(t) for t in parts[3:]]
                
                annotations[article_id].append({
                    'article_id': article_id,
                    'start_pos': start_pos,
                    'end_pos': end_pos,
                    'techniques': techniques,
                    'num_techniques': len(techniques)
                })
        
        print(f"✓ 加载了 {len(annotations)} 篇文章的标注")
        total_spans = sum(len(spans) for spans in annotations.values())
        print(f"✓ 总标注片段数: {total_spans}")
        
        return dict(annotations)
    
    def calculate_overlap(self, span1: Tuple[int, int], span2: Tuple[int, int]) -> float:
        """
        计算两个文本片段的重叠比例
        
        Args:
            span1: (start, end)
            span2: (start, end)
        
        Returns:
            重叠比例 (0.0 到 1.0)
        """
        start1, end1 = span1
        start2, end2 = span2
        
        # 计算重叠区域
        overlap_start = max(start1, start2)
        overlap_end = min(end1, end2)
        
        if overlap_start >= overlap_end:
            return 0.0  # 无重叠
        
        overlap_length = overlap_end - overlap_start
        
        # 计算相对于较短片段的重叠比例
        span1_length = end1 - start1
        span2_length = end2 - start2
        min_length = min(span1_length, span2_length)
        
        if min_length == 0:
            return 0.0
        
        return overlap_length / min_length
    
    def match_spans(
        self, 
        detected_spans: List[Dict], 
        gold_spans: List[Dict],
        overlap_threshold: float = 0.5
    ) -> Dict:
        """
        匹配检测片段和真实标注片段
        
        Args:
            detected_spans: 检测的片段列表
            gold_spans: 真实标注片段列表
            overlap_threshold: 重叠阈值（大于此值认为匹配）
        
        Returns:
            匹配结果字典
        """
        matched_pairs = []
        unmatched_detected = []
        unmatched_gold = []
        
        # 用于标记哪些gold spans已被匹配
        gold_matched = [False] * len(gold_spans)
        
        # 对每个检测片段，找最佳匹配的gold span
        for detected in detected_spans:
            detected_span = (detected['start_pos'], detected['end_pos'])
            
            best_overlap = 0.0
            best_gold_idx = -1
            
            for idx, gold in enumerate(gold_spans):
                if gold_matched[idx]:
                    continue  # 已被匹配
                
                gold_span = (gold['start_pos'], gold['end_pos'])
                overlap = self.calculate_overlap(detected_span, gold_span)
                
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_gold_idx = idx
            
            # 如果找到足够重叠的匹配
            if best_overlap >= overlap_threshold and best_gold_idx >= 0:
                matched_pairs.append({
                    'detected': detected,
                    'gold': gold_spans[best_gold_idx],
                    'overlap': best_overlap
                })
                gold_matched[best_gold_idx] = True
            else:
                unmatched_detected.append(detected)
        
        # 收集未匹配的gold spans
        for idx, gold in enumerate(gold_spans):
            if not gold_matched[idx]:
                unmatched_gold.append(gold)
        
        return {
            'matched_pairs': matched_pairs,
            'unmatched_detected': unmatched_detected,
            'unmatched_gold': unmatched_gold
        }
    
    def evaluate_techniques(self, matched_pairs: List[Dict]) -> Dict:
        """
        评估技术检测的准确性（在匹配的片段对上）
        
        Args:
            matched_pairs: 匹配的片段对列表
        
        Returns:
            评估指标
        """
        total_detected = 0
        total_gold = 0
        total_correct = 0
        
        technique_stats = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
        
        for pair in matched_pairs:
            detected_techs = set(pair['detected']['detected_techniques'])
            gold_techs = set(pair['gold']['techniques'])
            
            total_detected += len(detected_techs)
            total_gold += len(gold_techs)
            
            # 真阳性：既在检测结果中，又在真实标注中
            correct = detected_techs & gold_techs
            total_correct += len(correct)
            
            # 假阳性：在检测结果中，但不在真实标注中
            false_positive = detected_techs - gold_techs
            
            # 假阴性：在真实标注中，但不在检测结果中
            false_negative = gold_techs - detected_techs
            
            # 更新每个技术的统计
            for tech in correct:
                technique_stats[tech]['tp'] += 1
            
            for tech in false_positive:
                technique_stats[tech]['fp'] += 1
            
            for tech in false_negative:
                technique_stats[tech]['fn'] += 1
        
        # 计算总体指标
        precision = total_correct / total_detected if total_detected > 0 else 0.0
        recall = total_correct / total_gold if total_gold > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        return {
            'total_detected': total_detected,
            'total_gold': total_gold,
            'total_correct': total_correct,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'technique_stats': dict(technique_stats)
        }
    
    def evaluate_article(
        self,
        detected_result: Dict,
        gold_annotations: List[Dict],
        overlap_threshold: float = 0.5
    ) -> Dict:
        """
        评估单篇文章的检测结果
        
        Args:
            detected_result: 检测结果（来自detect_article_by_paragraphs）
            gold_annotations: 真实标注列表
            overlap_threshold: 片段重叠阈值
        
        Returns:
            评估结果
        """
        article_id = detected_result['article_id']
        
        # 提取检测的片段（只保留有技术的）
        detected_spans = [
            span for span in detected_result['paragraph_results']
            if span['num_techniques'] > 0
        ]
        
        # 匹配片段
        matching_result = self.match_spans(detected_spans, gold_annotations, overlap_threshold)
        
        # 评估技术
        technique_eval = self.evaluate_techniques(matching_result['matched_pairs'])
        
        # 片段级别的指标
        num_detected_spans = len(detected_spans)
        num_gold_spans = len(gold_annotations)
        num_matched_spans = len(matching_result['matched_pairs'])
        
        span_precision = num_matched_spans / num_detected_spans if num_detected_spans > 0 else 0.0
        span_recall = num_matched_spans / num_gold_spans if num_gold_spans > 0 else 0.0
        span_f1 = 2 * span_precision * span_recall / (span_precision + span_recall) if (span_precision + span_recall) > 0 else 0.0
        
        return {
            'article_id': article_id,
            'span_level': {
                'detected': num_detected_spans,
                'gold': num_gold_spans,
                'matched': num_matched_spans,
                'precision': span_precision,
                'recall': span_recall,
                'f1': span_f1
            },
            'technique_level': technique_eval,
            'matching_details': matching_result
        }
    
    def print_evaluation_report(self, eval_result: Dict, detailed: bool = True):
        """
        打印评估报告
        
        Args:
            eval_result: 评估结果
            detailed: 是否显示详细信息
        """
        article_id = eval_result['article_id']
        
        print(f"\n{'='*70}")
        print(f"评估报告: {article_id}")
        print(f"{'='*70}")
        
        # 片段级别指标
        span = eval_result['span_level']
        print(f"\n📍 片段级别评估:")
        print(f"  检测片段数: {span['detected']}")
        print(f"  真实片段数: {span['gold']}")
        print(f"  匹配片段数: {span['matched']}")
        print(f"  精确率: {span['precision']:.2%}")
        print(f"  召回率: {span['recall']:.2%}")
        print(f"  F1分数: {span['f1']:.2%}")
        
        # 技术级别指标
        tech = eval_result['technique_level']
        print(f"\n🎯 技术级别评估:")
        print(f"  检测技术数: {tech['total_detected']}")
        print(f"  真实技术数: {tech['total_gold']}")
        print(f"  正确技术数: {tech['total_correct']}")
        print(f"  精确率: {tech['precision']:.2%}")
        print(f"  召回率: {tech['recall']:.2%}")
        print(f"  F1分数: {tech['f1']:.2%}")
        
        # 详细的技术统计
        if detailed and tech['technique_stats']:
            print(f"\n📊 各技术详细统计:")
            print(f"{'技术名称':<35} {'TP':>4} {'FP':>4} {'FN':>4} {'精确率':>7} {'召回率':>7} {'F1':>7}")
            print(f"{'-'*70}")
            
            for tech_name, stats in sorted(tech['technique_stats'].items()):
                tp = stats['tp']
                fp = stats['fp']
                fn = stats['fn']
                
                prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
                
                print(f"{tech_name:<35} {tp:>4} {fp:>4} {fn:>4} {prec:>6.1%} {rec:>6.1%} {f1:>6.1%}")
        
        # 错误分析
        if detailed:
            matching = eval_result['matching_details']
            
            if matching['unmatched_detected']:
                print(f"\n⚠️  假阳性片段 ({len(matching['unmatched_detected'])} 个):")
                for span in matching['unmatched_detected'][:3]:  # 只显示前3个
                    print(f"  位置 {span['start_pos']}-{span['end_pos']}: {', '.join(span['detected_techniques'][:3])}")
            
            if matching['unmatched_gold']:
                print(f"\n⚠️  假阴性片段 ({len(matching['unmatched_gold'])} 个):")
                for span in matching['unmatched_gold'][:3]:  # 只显示前3个
                    print(f"  位置 {span['start_pos']}-{span['end_pos']}: {', '.join(span['techniques'][:3])}")
        
        print(f"{'='*70}\n")

# ══ 以下初始化代码已注释 ══
# 如需评估，请在独立 cell 中手动初始化 PropagandaEvaluator()
# 并加载正确的金标准文件（CheckThat 2024 开发集），而非此处的训练集标注
#
# 
# # ============================================================
# # 初始化评估器
# # ============================================================
# 
# print("="*70)
# print("🎯 初始化评估系统")
# print("="*70)
# 
# evaluator = PropagandaEvaluator()
# 
# print("✓ 评估器初始化完成\n")
# 
# 
# # ============================================================
# # 加载真实标注
# # ============================================================
# 
# print("="*70)
# print("📂 加载真实标注")
# print("="*70)
# 
# # 波兰语标注
# PL_ANNOTATION_FILE = "your/path/here"  # Update to your local path
# 
# pl_gold_annotations = evaluator.load_gold_annotations(PL_ANNOTATION_FILE)
# 
# if pl_gold_annotations:
#     total_spans = sum(len(spans) for spans in pl_gold_annotations.values())
#     total_techniques = sum(
#         sum(len(span['techniques']) for span in spans)
#         for spans in pl_gold_annotations.values()
#     )
#     
#     print(f"\n📊 标注统计:")
#     print(f"  文章数: {len(pl_gold_annotations)}")
#     print(f"  标注片段数: {total_spans}")
#     print(f"  技术实例数: {total_techniques}")
#     print(f"  平均每篇: {total_spans/len(pl_gold_annotations):.1f} 个片段")
#     print(f"  平均每片段: {total_techniques/total_spans:.1f} 个技术")
# 
# print("="*70)
# 
# 
# # # ============================================================
# # # 评估单篇文章（测试）
# # # ============================================================
# 
# # if 'result' in locals() and pl_gold_annotations:
# #     article_id = result['article_id']
#     
# #     if article_id in pl_gold_annotations:
# #         print("\n" + "="*70)
# #         print("🧪 评估测试文章")
# #         print("="*70)
#         
# #         eval_result = evaluator.evaluate_article(
# #             detected_result=result,
# #             gold_annotations=pl_gold_annotations[article_id],
# #             overlap_threshold=0.5
# #         )
#         
# #         evaluator.print_evaluation_report(eval_result, detailed=True)
# #     else:
# #         print(f"\n⚠️  测试文章 {article_id} 没有真实标注")
# #         print(f"   标注文件中的文章: {list(pl_gold_annotations.keys())[:5]}...")

## 批量测试函数

In [ ]:
# ============================================================
# 修改版 batch_detect_unified - 支持自定义输出路径
# ============================================================

import os
from pathlib import Path

def batch_detect_unified(
    articles_dict: Dict[str, Dict[str, str]],
    detector,
    detector_name: str = "检测器",
    languages_to_test: List[str] = None,
    num_articles_per_language: int = None,
    min_paragraph_length: int = 50,
    # 评估相关参数（可选）
    evaluator = None,
    gold_annotations: Dict[str, List] = None,
    overlap_threshold: float = 0.5,
    # 输出控制
    output_dir: str = None,  # 🆕 新增：输出目录
    save_tsv: bool = True,
    save_json: bool = True,
    verbose: bool = True
) -> Dict:
    """
    统一的批量检测函数 - 支持有评估和无评估两种模式
    
    Args:
        articles_dict: 文章字典
        detector: 检测器实例
        detector_name: 检测器名称
        languages_to_test: 要测试的语言列表
        num_articles_per_language: 每语言处理的文章数
        min_paragraph_length: 最小段落长度
        
        evaluator: 评估器实例
        gold_annotations: 金标准标注
        overlap_threshold: 片段重叠阈值
        
        output_dir: 输出目录路径（None表示当前目录）
        save_tsv: 是否保存TSV格式
        save_json: 是否保存JSON格式
        verbose: 是否显示详细信息
    
    Returns:
        所有语言的检测结果字典
    """
    import time
    import json
    
    # 🆕 创建输出目录
    if output_dir:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"\n📁 输出目录: {output_dir}")
    else:
        output_path = Path(".")
    
    # 判断是否为评估模式
    evaluation_mode = (evaluator is not None and gold_annotations is not None)
    
    # 语言配置映射
    language_display = {
        'ru': {'name': '俄语', 'flag': '🇷🇺'},
        'po': {'name': '波兰语', 'flag': '🇵🇱'},
        'pl': {'name': '波兰语', 'flag': '🇵🇱'},
        'en': {'name': '英语', 'flag': '🇬🇧'},
        'bg': {'name': '保加利亚语', 'flag': '🇧🇬'},
    }
    
    if languages_to_test is None:
        languages_to_test = list(articles_dict.keys())
    
    all_results = {}
    start_time = time.time()
    
    print("\n" + "="*70)
    mode_str = "批量检测与评估" if evaluation_mode else "批量检测（无评估模式）"
    print(f"🚀 {mode_str}: {detector_name}")
    print("="*70)
    print(f"测试语言: {', '.join(languages_to_test)}")
    print(f"每语言文章数: {'全部' if num_articles_per_language is None else num_articles_per_language}")
    if save_tsv:
        print(f"✓ 将保存TSV格式输出")
    if save_json:
        print(f"✓ 将保存JSON格式输出")
    print("="*70 + "\n")
    
    for idx, lang_code in enumerate(languages_to_test, 1):
        lang_info = language_display.get(lang_code, {'name': lang_code.upper(), 'flag': '🌐'})
        
        try:
            print(f"\n{'='*70}")
            print(f"{lang_info['flag']} [{idx}/{len(languages_to_test)}] 检测 {lang_info['name']} ({lang_code.upper()})")
            print(f"{'='*70}\n")
            
            lang_start_time = time.time()
            
            # 获取文章
            if lang_code not in articles_dict:
                print(f"❌ 没有找到语言 '{lang_code}' 的文章")
                all_results[lang_code] = {'error': f"没有找到语言 '{lang_code}' 的文章"}
                continue
            
            articles = articles_dict[lang_code]
            
            # 转换列表格式为字典
            if isinstance(articles, list):
                print(f"  ℹ️  检测到列表格式，正在转换为字典...")
                articles_dict_converted = {}
                for item in articles:
                    if isinstance(item, dict):
                        article_id = item.get('filename') or item.get('id') or item.get('article_id')
                        article_text = item.get('text') or item.get('content') or item.get('body')
                        if article_id and article_text:
                            articles_dict_converted[article_id] = article_text
                articles = articles_dict_converted
                print(f"  ✓ 转换完成: {len(articles)} 篇文章")
            
            # 限制文章数量
            if num_articles_per_language is not None and isinstance(articles, dict):
                articles_items = list(articles.items())[:num_articles_per_language]
                articles = dict(articles_items)
            
            if not articles:
                print(f"❌ 该语言没有文章可处理")
                all_results[lang_code] = {'error': '没有文章可处理'}
                continue
            
            print(f"✓ 成功加载 {len(articles)} 篇文章")
            
            # 检测所有文章
            print(f"\n🔍 开始检测...")
            
            article_results = []
            tsv_outputs = []
            
            for i, (article_id, article_text) in enumerate(articles.items(), 1):
                if verbose:
                    print(f"\n处理文章 {i}/{len(articles)}: {article_id}")
                
                try:
                    result = detector.detect_article_by_paragraphs(
                        article_id=article_id,
                        full_text=article_text,
                        min_paragraph_length=min_paragraph_length,
                        verbose=False
                    )
                    
                    article_results.append(result)
                    
                    if save_tsv:
                        tsv_lines = detector.format_output_tsv(result)
                        tsv_outputs.extend(tsv_lines)
                    
                    if verbose:
                        num_paragraphs = len(result['paragraph_results'])
                        num_with_tech = sum(1 for p in result['paragraph_results'] if p['num_techniques'] > 0)
                        total_techniques = sum(p['num_techniques'] for p in result['paragraph_results'])
                        print(f"  ✓ 检测完成: {num_with_tech}/{num_paragraphs} 段落有技术, 共 {total_techniques} 个技术实例")
                
                except Exception as e:
                    print(f"  ❌ 错误: {e}")
                    continue
            
            # 保存TSV 🆕 保存到指定目录
            if save_tsv and tsv_outputs:
                detector_name_safe = detector_name.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')
                tsv_filename = output_path / f"results_{lang_code}_{detector_name_safe}.tsv"
                
                with open(tsv_filename, 'w', encoding='utf-8') as f:
                    f.write("article_id\tstart_pos\tend_pos\ttechniques\n")
                    for line in tsv_outputs:
                        f.write(line + '\n')
                
                print(f"\n💾 TSV输出已保存: {tsv_filename}")
            
            # 保存JSON 🆕 保存到指定目录
            if save_json and article_results:
                detector_name_safe = detector_name.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')
                json_filename = output_path / f"results_{lang_code}_{detector_name_safe}_detailed.json"
                
                total_paragraphs = sum(len(r['paragraph_results']) for r in article_results)
                total_paras_with_tech = sum(
                    sum(1 for p in r['paragraph_results'] if p['num_techniques'] > 0)
                    for r in article_results
                )
                total_technique_instances = sum(
                    sum(p['num_techniques'] for p in r['paragraph_results'])
                    for r in article_results
                )
                
                technique_counter = {}
                for result in article_results:
                    for para_result in result['paragraph_results']:
                        for tech in para_result['detected_techniques']:
                            technique_counter[tech] = technique_counter.get(tech, 0) + 1
                
                json_data = {
                    'metadata': {
                        'language': lang_info['name'],
                        'language_code': lang_code,
                        'detector_name': detector_name,
                        'total_articles': len(article_results),
                        'total_paragraphs': total_paragraphs,
                        'paragraphs_with_techniques': total_paras_with_tech,
                        'total_technique_instances': total_technique_instances,
                        'technique_frequency': dict(sorted(technique_counter.items(), key=lambda x: x[1], reverse=True))
                    },
                    'articles': article_results
                }
                
                with open(json_filename, 'w', encoding='utf-8') as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=2)
                
                print(f"💾 JSON详细结果已保存: {json_filename}")
            
            # 统计摘要
            lang_elapsed = time.time() - lang_start_time
            
            if article_results:
                total_paragraphs = sum(len(r['paragraph_results']) for r in article_results)
                total_paras_with_tech = sum(
                    sum(1 for p in r['paragraph_results'] if p['num_techniques'] > 0)
                    for r in article_results
                )
                total_technique_instances = sum(
                    sum(p['num_techniques'] for p in r['paragraph_results'])
                    for r in article_results
                )
                
                technique_counter = {}
                for result in article_results:
                    for para_result in result['paragraph_results']:
                        for tech in para_result['detected_techniques']:
                            technique_counter[tech] = technique_counter.get(tech, 0) + 1
                
                summary = {
                    'language': lang_code,
                    'language_name': lang_info['name'],
                    'detector_name': detector_name,
                    'num_articles_processed': len(article_results),
                    'total_paragraphs': total_paragraphs,
                    'paragraphs_with_techniques': total_paras_with_tech,
                    'total_technique_instances': total_technique_instances,
                    'technique_frequency': dict(sorted(technique_counter.items(), key=lambda x: x[1], reverse=True)),
                    'elapsed_time': lang_elapsed
                }
                
                print(f"\n✓ {lang_info['name']}检测完成 (耗时: {lang_elapsed:.1f}秒):")
                print(f"  处理文章数: {len(article_results)}")
                print(f"  总段落数: {total_paragraphs}")
                print(f"  有技术的段落: {total_paras_with_tech} ({(total_paras_with_tech/total_paragraphs*100):.1f}%)")
                print(f"  总技术实例数: {total_technique_instances}")
                
                if technique_counter:
                    print(f"\n  🔝 最常见的技术 (Top 5):")
                    for tech, count in list(sorted(technique_counter.items(), key=lambda x: x[1], reverse=True))[:5]:
                        percentage = (count / total_technique_instances) * 100 if total_technique_instances > 0 else 0
                        print(f"    {tech}: {count} ({percentage:.1f}%)")
            else:
                summary = {'language': lang_code, 'error': '没有文章被成功处理'}
            
            all_results[lang_code] = summary
            
        except Exception as e:
            print(f"\n❌ {lang_info['name']}检测失败: {e}")
            all_results[lang_code] = {'error': str(e)}
    
    total_elapsed = time.time() - start_time
    
    print("\n" + "="*70)
    print(f"📊 所有语言检测总结 - {detector_name}")
    print("="*70)
    print(f"\n{'语言':<15} {'文章数':<8} {'总段落':<10} {'有技术段落':<12} {'技术实例':<10} {'耗时':<10}")
    print("-"*70)
    
    for lang_code in languages_to_test:
        if lang_code not in all_results:
            continue
        
        result = all_results[lang_code]
        lang_info = language_display.get(lang_code, {'name': lang_code.upper()})
        lang_name = lang_info['name']
        
        if 'error' in result:
            error_msg = result['error'][:20]
            print(f"{lang_name:<15} {'N/A':<8} {'N/A':<10} {'N/A':<12} {'N/A':<10} {error_msg:<10}")
        else:
            num_articles = result['num_articles_processed']
            total_paras = result.get('total_paragraphs', 0)
            paras_with_tech = result.get('paragraphs_with_techniques', 0)
            tech_instances = result.get('total_technique_instances', 0)
            elapsed = result.get('elapsed_time', 0)
            print(f"{lang_name:<15} {num_articles:<8} {total_paras:<10} {paras_with_tech:<12} {tech_instances:<10} {elapsed:<9.1f}s")
    
    print("-"*70)
    print(f"{'总耗时':<15} {'':<8} {'':<10} {'':<12} {'':<10} {total_elapsed:<9.1f}s")
    print("="*70 + "\n")
    
    return all_results


print("✓ 修改版 batch_detect_unified Functions defined（支持自定义输出路径）!")

In [ ]:
# ============================================================
# 多检测器对比测试函数
# ============================================================

def compare_detectors(
    data_base_dir: str,
    language_configs: Dict,
    detectors: Dict,  # {'名称': 检测器实例}
    evaluator: PropagandaEvaluator,
    languages_to_test: List[str] = None,
    num_articles_per_language: int = None,
    min_paragraph_length: int = 50,
    overlap_threshold: float = 0.5,
    save_tsv: bool = False
) -> Dict:
    """
    对比多个检测器的性能
    
    Args:
        detectors: 字典，格式为 {'检测器名称': 检测器实例}
        其他参数同 batch_test_all_languages
    
    Returns:
        所有检测器的结果字典
    """
    
    all_detector_results = {}
    
    print("\n" + "="*70)
    print(f"🔬 多检测器对比测试")
    print("="*70)
    print(f"检测器数量: {len(detectors)}")
    print(f"检测器列表: {', '.join(detectors.keys())}")
    print("="*70 + "\n")
    
    # 对每个检测器运行批量测试
    for detector_name, detector in detectors.items():
        print(f"\n{'#'*70}")
        print(f"# 测试检测器: {detector_name}")
        print(f"{'#'*70}\n")
        
        results = batch_test_all_languages(
            data_base_dir=data_base_dir,
            language_configs=language_configs,
            detector=detector,
            evaluator=evaluator,
            detector_name=detector_name,
            languages_to_test=languages_to_test,
            num_articles_per_language=num_articles_per_language,
            min_paragraph_length=min_paragraph_length,
            overlap_threshold=overlap_threshold,
            verbose=False,  # 对比模式下不显示详细信息
            save_tsv=save_tsv
        )
        
        all_detector_results[detector_name] = results
    
    # 打印综合对比表
    print("\n" + "="*70)
    print("📊 检测器性能综合对比")
    print("="*70)
    
    if languages_to_test is None:
        languages_to_test = list(language_configs.keys())
    
    for lang_code in languages_to_test:
        lang_name = language_configs[lang_code]['name']
        print(f"\n{lang_name}:")
        print(f"{'检测器':<25} {'片段F1':<12} {'技术F1':<12} {'检测数/金标':<15}")
        print("-"*70)
        
        for detector_name in detectors.keys():
            result = all_detector_results[detector_name].get(lang_code, {})
            
            if 'error' in result:
                print(f"{detector_name:<25} {'N/A':<12} {'N/A':<12} {'N/A':<15}")
            else:
                span_f1 = result.get('avg_span_f1', 0)
                tech_f1 = result.get('avg_tech_f1', 0)
                detected = result.get('total_detected_spans', 0)
                gold = result.get('total_gold_spans', 0)
                
                print(f"{detector_name:<25} {span_f1:<11.1%} {tech_f1:<11.1%} {detected}/{gold:<13}")
    
    print("="*70 + "\n")
    
    return all_detector_results


print("✓ 多检测器对比Functions defined!")

## 加载真实文章进行测试

In [ ]:
# ============================================================
# 语言显示信息（仅用于打印，不影响检测逻辑）
# ============================================================
LANGUAGE_INFO = {
    'en': {'name': '英语',   'flag': '🇺🇸'},
    'po': {'name': '波兰语', 'flag': '🇵🇱'},
    'ru': {'name': '俄语',   'flag': '🇷🇺'},
}


def load_articles_from_path(article_dir: str, language: str = None, label: str = None):
    """
    从指定路径加载文章原文

    Args:
        article_dir: 文章目录路径（每个 .txt 文件为一篇文章）
        language:    语言代码，如 'en' / 'po' / 'ru'（可选，用于打印）
        label:       数据集标签，如 'en' / 'po'（可选，作为返回字典的键）
    """
    if not os.path.exists(article_dir):
        print(f"✗ 目录不存在: {article_dir}")
        return None

    lang_info = LANGUAGE_INFO.get(language, {})
    print(f"\n{'='*70}")
    print(f"📂 加载路径: {article_dir}")
    if language:
        print(f"🌍 语言: {lang_info.get('flag', '')} {lang_info.get('name', language)}")
    if label:
        print(f"🏷️  标签: {label}")
    print(f"{'='*70}")

    all_txt = list(Path(article_dir).glob('*.txt'))
    txt_files = [f for f in all_txt if not f.name.startswith('.')]

    if not txt_files:
        print(f"✗ 未找到任何有效的 .txt 文件")
        return None

    hidden = len(all_txt) - len(txt_files)
    print(f"📄 找到 {len(txt_files)} 个有效文本文件" +
          (f" (已过滤 {hidden} 个隐藏文件)" if hidden else ""))

    articles = []
    failed = 0
    for f in txt_files:
        content = None
        for enc in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
            try:
                content = f.read_text(encoding=enc).strip()
                break
            except UnicodeDecodeError:
                continue
        if content is None:
            print(f"  ⚠️ 无法解码: {f.name}")
            failed += 1
            continue
        if content:
            articles.append({
                'filename':   f.name,
                'text':       content,
                'language':   language or 'unknown',
                'label':      label or 'unlabeled',
                'char_count': len(content),
                'word_count': len(content.split())
            })

    print(f"✓ 成功加载 {len(articles)} 篇文章" + (f" (失败: {failed})" if failed else ""))

    if articles:
        total_chars = sum(a['char_count'] for a in articles)
        total_words = sum(a['word_count'] for a in articles)
        print(f"\n📊 数据统计:")
        print(f"  - 文章数量: {len(articles)}")
        print(f"  - 总字符数: {total_chars:,}")
        print(f"  - 总词数:   {total_words:,}")
        print(f"  - 平均字符/篇: {total_chars/len(articles):.0f}")
        print(f"  - 平均词数/篇: {total_words/len(articles):.0f}")

    return articles


def load_multiple_sources(sources: list) -> dict:
    """
    从多个来源加载文章

    Args:
        sources: 列表，每个元素为 {'path': ..., 'language': ..., 'label': ...}

    Returns:
        dict: {label: [articles]}
    """
    all_articles = {}
    for source in sources:
        articles = load_articles_from_path(
            source['path'],
            language=source.get('language'),
            label=source.get('label', 'default')
        )
        if articles:
            all_articles[source.get('label', 'default')] = articles
    return all_articles


print("✓ 文章加载Functions defined")


## 开始测试

In [ ]:
# ══ 已注释 [示例-单语言测试 → 请使用下方稳定版分批处理] ══
# # 一个示例
# # ============================================================
# # 测试俄语和波兰语（使用正确的检测器）
# # ============================================================
# 
# # # 准备数据
# # test_data_sources = [
# #     {
# #         'path': 'your/path/here'  # CheckThat! 2024 data directory,
# #         'language': 'ru',
# #         'label': 'ru'
# #     },
# #     {
# #         'path': 'your/path/here'  # CheckThat! 2024 data directory,
# #         'language': 'po',
# #         'label': 'po'
# #     }
# # ]
# 
# # # Load data
# # test_articles = load_multiple_sources(test_data_sources)
# 
# # ============================================================
# # 测试俄语（使用俄语检测器）
# # ============================================================
# 
# print("\n" + "🇷🇺"*35)
# print("测试俄语文章 - 使用俄语检测器")
# print("🇷🇺"*35 + "\n")
# 
# results_ru = batch_detect_unified(
#     # ============================================================
#     # 核心参数（必需）
#     # ============================================================
#     
#     articles_dict={'ru': test_articles['ru']},
#     # 📚 文章数据字典
#     # 格式：{语言代码: 文章列表/字典}
#     
#     detector=context_detector_ru,
#     # 🔍 检测器对象
#     # 用哪个检测器来检测propaganda
#     # 必须匹配语言！俄语文章用俄语检测器
#     # 可选：
#     # context_detector_po                  # 普通检测器
#     # voting_detector_po_balanced          # 投票-平衡
#     # voting_detector_po_conservative      # 投票-保守  
#     # voting_detector_po_aggressive        # 投票-激进
# 
#     # # 向后兼容的别名（指向波兰语检测器）
#     # context_detector                     # = context_detector_po
#     # voting_detector_balanced             # = voting_detector_po_balanced
#     # voting_detector_conservative         # = voting_detector_po_conservative
#     # voting_detector_aggressive           # = voting_detector_po_aggressive
#     
#     # ============================================================
#     # 配置参数
#     # ============================================================
#     
#     detector_name="results_po_checkthat2024_dev_Ordinary_strict",
#     # - 用于生成输出文件名
#     # 例如：results_ru_俄语普通检测器.tsv
#     
#     languages_to_test=['ru'],
#     # 🌍 要测试的语言列表
#     # 必须匹配 articles_dict 中的键
#     
#     num_articles_per_language=1,
#     # 📊 每种语言处理多少篇文章
#     # - 10: 只处理前10篇（用于快速测试）
#     # - None: 处理全部文章（用于完整测试）
#     
#     # ============================================================
#     # 输出控制参数
#     # ============================================================
#     
#     save_tsv=True,
#     # 💾 是否保存TSV格式的结果
#     # True: 生成 results_ru_俄语普通检测器.tsv
#     # TSV格式：article_id    start_pos    end_pos    techniques
#     
#     save_json=True,
#     # 💾 是否保存JSON格式的详细结果
#     # True: 生成 results_ru_俄语普通检测器_detailed.json
#     # 包含：元数据、所有文章的完整检测结果、技术频率统计
#     
#     verbose=True,
#     # 📢 是否显示详细的进度信息
#     # True: 显示每篇文章的处理进度和结果
#     #   "处理文章 1/10: article3428.txt"
#     #   "✓ 检测完成: 3/13 段落有技术, 共 8 个技术实例"
#     # False: 只显示摘要信息
#     
#     # ============================================================
#     # 可选参数（用于评估模式）
#     # ============================================================
#     
#     # evaluator=None,
#     # 🎯 评估器对象（默认 None = 无评估模式）
#     # 如果提供：进行F1/精确率/召回率评估
#     # None：只检测，不评估
#     
#     # gold_annotations=None,
#     # 📋 金标准标注数据（默认 None）
#     # 格式：{'ru': {article_id: [标注列表]}}
#     # 只有在评估模式才需要
#     
#     # min_paragraph_length=50,
#     # 📏 最小段落长度（字符数）
#     # 小于这个长度的段落会被跳过
#     # 默认：50
#     
#     # overlap_threshold=0.5,
#     # 🎚️ 评估时的重叠阈值
#     # 只在评估模式有效
#     # 默认：0.5
# )
# 
# 
# # # ============================================================
# # # 结果对比
# # # ============================================================
# 
# # print("\n" + "="*70)
# # print("📊 检测结果对比")
# # print("="*70)
# 
# # print(f"\n{'语言':<15} {'文章数':<10} {'技术实例':<10} {'有技术段落':<15}")
# # print("-"*50)
# 
# # if 'ru' in results_ru and 'error' not in results_ru['ru']:
# #     r = results_ru['ru']
# #     print(f"{'俄语':<15} {r['num_articles_processed']:<10} {r['total_technique_instances']:<10} {r['paragraphs_with_techniques']}/{r['total_paragraphs']}")
# 
# # if 'po' in results_po and 'error' not in results_po['po']:
# #     r = results_po['po']
# #     print(f"{'波兰语':<15} {r['num_articles_processed']:<10} {r['total_technique_instances']:<10} {r['paragraphs_with_techniques']}/{r['total_paragraphs']}")
# 
# # print("="*70)

In [ ]:
# 准备数据
test_data_sources = [
    {
        'path': 'your/path/here'  # SemEval 2023 data directory,
        'language': 'ru',
        'label': 'ru'
    },
    {
        'path': 'your/path/here'  # SemEval 2023 data directory,
        'language': 'po',
        'label': 'po'
    },

    {
        'path': 'your/path/here'  # SemEval 2023 data directory,
        'language': 'en',
        'label': 'en'
    }
]

# Load data
test_articles = load_multiple_sources(test_data_sources)

## 稳定版分批处理（正式运行入口）

> **只需修改下方配置 cell 中的 `LANGUAGE` 和 `DETECTOR_TYPE`，然后依次运行两个 cell。**

| 变量 | 可选值 | 说明 |
|---|---|---|
| `LANGUAGE` | `'en'` / `'po'` / `'ru'` | 目标语言 |
| `DETECTOR_TYPE` | `'context'` | Prompt-A：普通检测器（快速） |
| | `'voting_aggressive'` | Iter-Ens：激进（threshold=0.34，召回率高） |
| | `'voting_balanced'` | Iter-Ens：平衡（threshold=0.67） |
| | `'voting_conservative'` | Iter-Ens：保守（threshold=0.80，精确率高） |
| `BATCH_SIZE` | 整数（推荐24） | 每批文章数 |
| `NUM_BATCHES` | 整数 | 总批次数（= ceil(总文章数 / BATCH_SIZE)） |

**继续上次中断**: 将运行 cell 中的 `for batch in range(1, NUM_BATCHES + 1)` 改为 `for batch in range(起始批次, NUM_BATCHES + 1)` 即可。

In [ ]:
# ============================================================
# 稳定版分批处理 - 显示文章ID + 使用原始检测函数
# ============================================================

import time
from datetime import datetime

# ============================================================
# 🎯 只需修改这两行!
# ============================================================

LANGUAGE = 'po'      # 可选: 'ru', 'po', 'en'
DETECTOR_TYPE = 'voting_aggressive'  # 可选: 'context', 'voting_aggressive激进', 'voting_balanced', 'voting_conservative'
BATCH_SIZE = 24
NUM_BATCHES = 4  # 批次数量
SLEEP_TIME = 1

# ============================================================
# 自动配置(不需要修改)
# ============================================================

# 检测器配置
DETECTOR_CONFIG = {
    'ru': {
        'context': context_detector_ru,
        'voting_aggressive': voting_detector_ru_aggressive,
        'voting_balanced': voting_detector_ru_balanced,
        'voting_conservative': voting_detector_ru_conservative
    },
    'po': {
        'context': context_detector_po,
        'voting_aggressive': voting_detector_po_aggressive,
        'voting_balanced': voting_detector_po_balanced,
        'voting_conservative': voting_detector_po_conservative
    },
    'en': {
        'context': context_detector_en,
        'voting_aggressive': voting_detector_en_aggressive,
        'voting_balanced': voting_detector_en_balanced,
        'voting_conservative': voting_detector_en_conservative
    }
}

# 语言信息
LANGUAGE_INFO = {
    'ru': {'name': '俄语', 'flag': '🇷🇺'},
    'po': {'name': '波兰语', 'flag': '🇵🇱'},
    'en': {'name': '英语', 'flag': '🇺🇸'}  # 或用 🇺🇸
}

# 检测器信息
DETECTOR_INFO = {
    'context': {'name': '普通检测器', 'icon': '🔍'},
    'voting_aggressive': {'name': '投票检测器(激进)', 'icon': '🚀'},
    'voting_balanced': {'name': '投票检测器(平衡)', 'icon': '⚖️'},
    'voting_conservative': {'name': '投票检测器(保守)', 'icon': '🛡️'}
}

# 固定配置
OUTPUT_DIR = "your/path/here"  # Update to your local path

# ============================================================
# 数据键名映射(自动检测)
# ============================================================

# 检测使用哪个变量和键名
data_var = None
data_key = None

if 'all_data' in dir():
    data_var = all_data
    # all_data 使用 'ru_dev', 'po_dev', 'en_dev' 作为键
    if f'{LANGUAGE}_dev' in all_data:
        data_key = f'{LANGUAGE}_dev'
    elif LANGUAGE in all_data:
        data_key = LANGUAGE
elif 'test_articles' in dir():
    data_var = test_articles
    # test_articles 使用 'ru', 'po', 'en' 作为键
    data_key = LANGUAGE

if data_var is None or data_key is None:
    print("❌ 错误: 找不到文章数据")
    print("\n可用变量:")
    if 'all_data' in dir():
        print(f"  all_data 的键: {list(all_data.keys())}")
    if 'test_articles' in dir():
        print(f"  test_articles 的键: {list(test_articles.keys())}")
    raise ValueError("文章数据未加载或键名不匹配")

In [ ]:
# ============================================================
# 运行（不需要修改）
# ============================================================

# 验证配置
if LANGUAGE not in DETECTOR_CONFIG:
    print(f"❌ 错误: 不支持的语言 '{LANGUAGE}'")
    print(f"   可选语言: {list(DETECTOR_CONFIG.keys())}")
    raise ValueError(f"不支持的语言: {LANGUAGE}")

if DETECTOR_TYPE not in DETECTOR_CONFIG[LANGUAGE]:
    print(f"❌ 错误: 不支持的检测器类型 '{DETECTOR_TYPE}'")
    print(f"   可选检测器: {list(DETECTOR_CONFIG[LANGUAGE].keys())}")
    raise ValueError(f"不支持的检测器: {DETECTOR_TYPE}")

# 获取配置
detector = DETECTOR_CONFIG[LANGUAGE][DETECTOR_TYPE]
lang_info = LANGUAGE_INFO[LANGUAGE]
detector_info = DETECTOR_INFO[DETECTOR_TYPE]

# 获取数据
articles = data_var[data_key]
total = len(articles)

# 显示配置
print("="*70)
print(f"🚀 稳定版分批处理 - 显示文章ID")
print("="*70)
print(f"数据源: {data_key} (共 {total} 篇)")
print(f"语言: {lang_info['flag']} {lang_info['name']} ({LANGUAGE})")
print(f"检测器: {detector_info['icon']} {detector_info['name']}")
print(f"批次配置: {NUM_BATCHES} 批 × {BATCH_SIZE} 篇/批")
print(f"输出目录: {OUTPUT_DIR}")
print("="*70 + "\n")

# 分批处理
all_results = []
start_time = time.time()

for batch in range(1, NUM_BATCHES + 1):
    start_idx = (batch - 1) * BATCH_SIZE
    end_idx = min(batch * BATCH_SIZE, total)
    batch_articles = articles[start_idx:end_idx]
    
    print(f"\n{lang_info['flag']} 批次 {batch}/{NUM_BATCHES}: 文章 {start_idx+1}-{end_idx} (共{len(batch_articles)}篇)")
    print(f"⏰ 开始: {datetime.now().strftime('%H:%M:%S')}")
    
    # 🆕 显示文章列表（不影响检测）
    print(f"\n📋 即将处理的文章:")
    for i, article in enumerate(batch_articles):
        article_id = article.get('filename', article.get('id', f'article_{start_idx + i + 1}'))
        if article_id.endswith('.txt'):
            article_id = article_id[:-4]
        print(f"   {i+1:2d}. {article_id}")
    
    print(f"\n🚀 开始批量检测...")
    
    # 使用原始的稳定检测函数
    result = batch_detect_unified(
        articles_dict={LANGUAGE: batch_articles},
        detector=detector,
        detector_name=f"results_semeval_task3_dev_{LANGUAGE}_{DETECTOR_TYPE}_batch{batch}",
        languages_to_test=[LANGUAGE],
        output_dir=OUTPUT_DIR,
        save_tsv=True,
        save_json=True,
        verbose=False
    )
    
    all_results.append(result)
    
    # 显示结果
    if LANGUAGE in result and 'error' not in result[LANGUAGE]:
        r = result[LANGUAGE]
        elapsed = time.time() - start_time
        avg_time_per_batch = elapsed / batch
        remaining_time = avg_time_per_batch * (NUM_BATCHES - batch)
        
        print(f"\n✅ 批次 {batch} 完成!")
        print(f"   📊 文章: {r['num_articles_processed']}, 技术: {r['total_technique_instances']}, 有技术段落: {r['paragraphs_with_techniques']}/{r['total_paragraphs']}")
        print(f"   ⏱️  本批耗时: {r['elapsed_time']/60:.1f} 分钟")
        print(f"   📈 总进度: {batch}/{NUM_BATCHES} ({batch/NUM_BATCHES*100:.0f}%)")
        
        if batch < NUM_BATCHES:
            print(f"   🔮 预计剩余: {remaining_time/60:.1f} 分钟")
        
        # 🆕 显示处理完成的文章确认
        print(f"   ✅ 已完成文章: {', '.join([article.get('filename', article.get('id', f'article_{i}')).replace('.txt', '') for i, article in enumerate(batch_articles)])}")
        
    else:
        print(f"\n❌ 批次 {batch} 失败")
        if 'error' in result.get(LANGUAGE, {}):
            print(f"   错误: {result[LANGUAGE]['error']}")
    
    # 批次间休息
    if batch < NUM_BATCHES and SLEEP_TIME > 0:
        print(f"\n⏸️  批次间休息 {SLEEP_TIME} 秒...")
        for i in range(SLEEP_TIME, 0, -1):
            if i % 10 == 0 or i <= 5:
                print(f"   ⏳ 剩余 {i} 秒...")
            time.sleep(1)

# 汇总
total_time = time.time() - start_time
successful = [r for r in all_results if LANGUAGE in r and 'error' not in r[LANGUAGE]]

print("\n" + "="*70)
print(f"🎉 处理完成! {lang_info['flag']} {lang_info['name']} - {detector_info['icon']} {detector_info['name']}")
print("="*70)

if successful:
    total_articles = sum(r[LANGUAGE]['num_articles_processed'] for r in successful)
    total_tech = sum(r[LANGUAGE]['total_technique_instances'] for r in successful)
    
    print(f"\n📊 最终统计:")
    print(f"  ✅ 成功批次: {len(successful)}/{NUM_BATCHES}")
    print(f"  📝 处理文章: {total_articles} 篇")
    print(f"  🎯 检测技术: {total_tech} 个")
    print(f"  ⏱️  总耗时: {total_time/60:.1f} 分钟 ({total_time/3600:.2f} 小时)")
    print(f"  🚀 平均速度: {total_time/total_articles:.1f} 秒/篇")
    
    print(f"\n💾 输出文件: {OUTPUT_DIR}/")
    for i in range(1, len(successful) + 1):
        print(f"    📄 批次{i}: results_semeval_task3_dev_{LANGUAGE}_{DETECTOR_TYPE}_batch{i}.tsv / .json") # 生成的文件名
else:
    print(f"\n❌ 所有批次都失败了")
    print(f"   请检查检测器配置和数据格式")

print("\n" + "="*70)

# ============================================================
# 📖 使用说明
# ============================================================

"""
🎯 使用方法 - 只需修改两行！

1. 选择语言:
   LANGUAGE = 'ru'    # 俄语
   LANGUAGE = 'po'    # 波兰语

2. 选择检测器:
   DETECTOR_TYPE = 'context'              # 普通检测器（快速）
   DETECTOR_TYPE = 'voting_aggressive'    # 投票-激进（召回率高）
   DETECTOR_TYPE = 'voting_balanced'      # 投票-平衡（推荐）
   DETECTOR_TYPE = 'voting_conservative'  # 投票-保守（精确率高）
"""

In [ ]:
# ══ 已注释 [旧版-无评估批量检测 → 已被 batch_detect_unified 替代] ══
# print("\n" + "🟢"*35)
# print("测试投票检测器 - 无评估模式")
# print("🟢"*35 + "\n")
# 
# # 配置数据源
# test_data_sources = [
#     {
#         'path': 'your/path/here'  # CheckThat! 2024 data directory,
#         'language': 'po',
#         'label': 'po'  # 改成po，不是po_dev
#     },
#     {
#         'path': 'your/path/here'  # CheckThat! 2024 data directory,
#         'language': 'ru',
#         'label': 'ru'  # 改成ru，不是ru_dev
#     },
# ]
# 
# # Load data
# test_articles = load_multiple_sources(test_data_sources)
# 
# # 使用无评估版本 + 投票检测器
# results_voting = batch_detect_without_evaluation(
#     articles_dict=test_articles,
#     detector=voting_detector_aggressive,
#     detector_name="checkthat2024_dev_Voting_aggressive",
#     languages_to_test=['ru', 'po'],  # 现在用ru和po
#     num_articles_per_language=5,  # 先测试5篇
#     min_paragraph_length=50,
#     verbose=True,
#     save_tsv=True,
#     save_json=True
# )

In [ ]:
# ══ 已注释 [示例-方式2旧调用（batch_test_all_languages，3个重复cell之1）] ══
# # ============================================================
# # 方式2: 单独测试投票检测器
# # ============================================================
# 
# print("\n" + "🟢"*35)
# print("测试投票检测器 (VotingContextAwareParagraphDetector)")
# print("🟢"*35 + "\n")
# 
# results_voting = batch_test_all_languages(
#     data_base_dir=DATA_BASE_DIR,
#     language_configs=LANGUAGE_CONFIGS,
#     detector=voting_detector_balanced,  # 投票检测器，voting_detector_aggressive（激进），voting_detector_conservative（保守），voting_detector_balanced（平衡）
#     evaluator=evaluator,
#     detector_name="投票检测器(平衡模式)",
#     languages_to_test=['ru', 'po'],
#     num_articles_per_language=None,
#     verbose=True,
#     save_tsv=True
# )

In [ ]:
# ══ 已注释 [示例-方式2旧调用（batch_test_all_languages，3个重复cell之2）] ══
# # ============================================================
# # 方式2: 单独测试投票检测器
# # ============================================================
# 
# print("\n" + "🟢"*35)
# print("测试投票检测器 (VotingContextAwareParagraphDetector)")
# print("🟢"*35 + "\n")
# 
# results_voting = batch_test_all_languages(
#     data_base_dir=DATA_BASE_DIR,
#     language_configs=LANGUAGE_CONFIGS,
#     detector=voting_detector_aggressive,  # 投票检测器，voting_detector_aggressive（激进），voting_detector_conservative（保守），voting_detector_balanced（平衡）
#     evaluator=evaluator,
#     detector_name="投票检测器(激进模式)",
#     languages_to_test=['ru', 'po'],
#     num_articles_per_language=None,
#     verbose=True,
#     save_tsv=True
# )

In [ ]:
# ══ 已注释 [示例-方式2旧调用（batch_test_all_languages，3个重复cell之3）] ══
# # ============================================================
# # 方式2: 单独测试投票检测器
# # ============================================================
# 
# print("\n" + "🟢"*35)
# print("测试投票检测器 (VotingContextAwareParagraphDetector)")
# print("🟢"*35 + "\n")
# 
# results_voting = batch_test_all_languages(
#     data_base_dir=DATA_BASE_DIR,
#     language_configs=LANGUAGE_CONFIGS,
#     detector=voting_detector_conservative,  # 投票检测器，voting_detector_aggressive（激进），voting_detector_conservative（保守），voting_detector_balanced（平衡）
#     evaluator=evaluator,
#     detector_name="投票检测器(保守模式)",
#     languages_to_test=['ru', 'po'],
#     num_articles_per_language=None,
#     verbose=True,
#     save_tsv=True
# )

In [ ]:
# ══ 已注释 [示例-方式3多检测器对比 → 调试用，非正式流程] ══
# # ============================================================
# # 方式3: 对比多个检测器（推荐！）
# # ============================================================
# 
# print("\n" + "🎯"*35)
# print("多检测器性能对比")
# print("🎯"*35 + "\n")
# 
# # 定义要对比的检测器
# detectors_to_compare = {
#     '普通检测器(严格)': context_detector,
#     '投票-平衡模式': voting_detector_balanced,
#     '投票-保守模式': voting_detector_conservative,
#     '投票-激进模式': voting_detector_aggressive
# }
# 
# # 运行对比测试
# comparison_results = compare_detectors(
#     data_base_dir=DATA_BASE_DIR,
#     language_configs=LANGUAGE_CONFIGS,
#     detectors=detectors_to_compare,
#     evaluator=evaluator,
#     languages_to_test=['ru', 'po'],  # 指定语言
#     num_articles_per_language=None,  # 每语言3篇文章
#     min_paragraph_length=50,
#     overlap_threshold=0.5,
#     save_tsv=True  # 为每个检测器保存TSV
# )
# 
# # 保存对比结果
# import json
# with open('detector_comparison_results.json', 'w', encoding='utf-8') as f:
#     # 只保存统计信息
#     summary = {}
#     for detector_name, lang_results in comparison_results.items():
#         summary[detector_name] = {
#             lang: {k: v for k, v in results.items() 
#                    if k not in ['article_results', 'article_evaluations']}
#             for lang, results in lang_results.items()
#         }
#     json.dump(summary, f, ensure_ascii=False, indent=2)
# 
# print("\n✓ 对比结果已保存到 detector_comparison_results.json")

## 查看详细结果

In [ ]:
# ══ 已注释 [示例-方式4 CSV导出 → 需单独调用 export_detailed_results_to_csv] ══
# # 方式4: 导出到CSV（可以在Excel中查看）
# print("\n💾 方式4: 导出到CSV")
# export_detailed_results_to_csv(results_normal, 'ru', 'results_ru_details.csv')
# export_detailed_results_to_csv(results_normal, 'po', 'results_po_details.csv')
# 
# print("\n✓ 可以用Excel打开这些CSV文件查看完整的段落和技术详情")

## 步骤15: 保存配置（可选）

In [ ]:
# ══ 已注释 [可选-步骤15保存配置（调用 loader.save_config）] ══
# # 保存当前配置
# config_output_path = "./flexible_system_config.json"
# 
# loader.save_config(config_output_path)
# 
# print(f"\n下次可以使用这个配置文件快速加载:")
# print(f"loader = FlexiblePromptLoader(base_dir=PROMPTS_BASE_DIR, config_file='{config_output_path}')")